# Migrating to a recent ns-3

**Decision for you: nothing here needs doing for Mode 2 to work.** This notebook gives what you need to decide.

## Verdict

**Migration is possible, and the code gap is small. The real risk is unnoticed change in Mode 2 results, and it can be controlled.** Stay on ns-3.22 unless you need MoReV2X in the same tree as your groupcast work.

Latest release on 2026-11-06: **ns-3.49**. MoReV2X is on ns-3.22, from February 2015.

| | Evidence |
|---|---|
| Possible? | 64 of 79 sources already type-check against ns-3.49, and the 70 errors in the rest have known fixes |
| Blocking stock patches? | None. Upstream's patches to stock ns-3 are debug logging, and Mode 2 needs none of them |
| Dependency on stock LTE? | None in the source: no stock `lte-*` header is included |
| Clash with 5G-LENA? | No class name collides, across 404 MoReV2X classes |
| Main risk | Results change without any error. Addressed with statistical comparison before and after |

**What was not done:** nothing was built, linked or run on a new ns-3. This is a compile-only trial. It finds API breaks, not behaviour changes, so it says the port is feasible, not that it works.

## The trial

Each source of `src/MoReV2X`, `src/traci` and `scratch/HIGHWAY.cc` was type-checked with `clang++ -fsyntax-only`. Nothing from either tree was executed, and nothing was written to the groupcast tree.

| Target | Standard | Files with errors | Errors |
|---|---|---|---|
| ns-3.42, your groupcast tree | C++20 | 12 of 79 | 48 |
| ns-3.49, latest | C++23 | 15 of 79 | 70 |
| ns-3.49, starting from carlosrisma's 3.36 port | C++23 | 13 of 79 | 55 |

Raw errors are in `docs/notebooks/data/`, and the script in `tools/ns3_compile_trial.py`. The 3.49 count includes 6 errors from the trial's own header stubs.

**carlosrisma's port saves only 15 errors.** It is not a useful starting point: you would inherit its rename and lose this repository's changes.

In [1]:
import os, subprocess, sys
sys.path.insert(0, os.path.abspath("../../tools"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 90)
mr.apply_chart_style()
os.chdir(mr.ROOT)

In [2]:
import collections, re

RULES = [
    ("Ptr compared with 0", r"Ptr<.*'(int|long)'"),
    ("EventId::IsRunning renamed IsPending", r"IsRunning"),
    ("GetRxAntenna renamed, returns Ptr<Object>", r"GetRxAntenna|AntennaModel \\*"),
    ("UdpClient::m_peerAddress renamed", r"m_peerAddress"),
    ("Angles fields became getters", r"'phi'|'theta'"),
    ("SpectrumPhy pure virtuals", r"abstract class"),
    ("Object rules: CreateObject, final GetInstanceTypeId", r"CreateObject|GetInstanceTypeId"),
    ("BuildingsHelper call removed", r"MakeMobilityModelConsistent"),
    ("MakeEnumAccessor needs a template argument", r"MakeEnumAccessor"),
    ("SocketAddressTag removed", r"SocketAddressTag"),
    ("Logging of a NIST type", r"basic_ostream"),
    ("Artefact of the trial headers", r"redefinition of"),
]
def classify(path):
    counts = collections.Counter()
    for line in open(path):
        if " error: " not in line:
            continue
        message = line.split(" error: ", 1)[1]
        counts[next((name for name, rule in RULES if re.search(rule, message)), "other")] += 1
    return counts

table = pd.DataFrame({version: classify(f"docs/notebooks/data/ns3-trial-{version}-errors.txt") for version in ("3.42", "3.49")})
table.fillna(0).astype(int).sort_values("3.49", ascending=False)

,3.42,3.49
Ptr compared with 0,34,34
EventId::IsRunning renamed IsPending,0,9
"GetRxAntenna renamed, returns Ptr<Object>",7,7
Artefact of the trial headers,0,6
UdpClient::m_peerAddress renamed,0,4
"Object rules: CreateObject, final GetInstanceTypeId",0,3
Angles fields became getters,2,2
SpectrumPhy pure virtuals,1,1
SocketAddressTag removed,1,1
Logging of a NIST type,1,1


## Every break, and its fix

| Break | Mode 2 path? | Fix |
|---|---|---|
| `Ptr == 0`, no longer compiles | Yes, also in `nr-v2x-spectrum-phy.cc` | Mechanical: `if (!ptr)` |
| `EventId::IsRunning` | Yes, 2 in `nr-v2x-rlc-um.cc` | Rename to `IsPending` |
| `GetRxAntenna`, `SpectrumPhy` pure virtuals | **Yes**, `NrV2XSpectrumPhy` becomes abstract | Add `GetAntenna`, and make `GetDevice` and `GetMobility` const, as carlosrisma did |
| `Angles.phi`, `.theta` | Antenna model | `GetAzimuth`, `GetInclination`. Check the angle convention against the old values |
| `m_peerAddress` in `UdpClient` | **Yes**, the scenario's traffic | See risk 2 |
| `Create<LTENodeState>`, `GetInstanceTypeId` | Scenario | `CreateObject`, delete the override |
| `BuildingsHelper::MakeMobilityModelConsistent` | Scenario | Delete the call |
| `MakeEnumAccessor` | `nist-lte-amc.cc` | Add the template argument |
| `SocketAddressTag` | No, EPC | Comment out, as carlosrisma did. Out of scope |

## Risks a type-check cannot see

Ordered by how much they matter for Mode 2.

| # | Risk | Severity | How to address it |
|---|---|---|---|
| 1 | **Silent drift of Mode 2 results.** `MultiModelSpectrumChannel`, the interference chain and event ordering have 10 years of changes. The code would run and give different numbers. RNG order changes too, so `tools/fingerprint.sh` cannot cross versions | **High** | Compare distributions, not hashes. Before porting, run a `tools/run_sweep.py` baseline on 3.22 with 10 or more `runNo` across loads and modes. After porting, PDR by distance, collisions, CBR and latency must fall inside the confidence intervals. A difference is a finding to explain before using the port |
| 2 | **The `UdpClient::Send` override.** `HIGHWAY` redefines a stock method, and needed `weak` patches in stock files on macOS. It would have to be redone in every version | Medium, certain | Replace it with a `V2xTrafficApplication` class of ours. This can be done on 3.22 first, where `fingerprint.sh` proves it changes nothing. Worth doing even without migrating |
| 3 | **Runtime-only names.** `ns3::DropTailQueue::MaxPackets` no longer exists, and `Config::SetDefault` aborts at startup | Low | Delete the line or use `MaxSize`. Runs fail loudly |
| 4 | **New assertions** in debug builds on code that relies on loose behaviour | Medium | `tools/debug-check.sh` covers 18 modes. It found 2 real bugs on 3.22 |
| 5 | **Undefined behaviour hidden by old compilers.** `Mode2Step1` uses `erase` followed by `--` on `std::list` iterators, which only works at `begin()` because libc++ lists are circular. Uninitialised members have already been found | Medium | Run the scenarios once under AddressSanitizer and UBSan. Newer ns-3 and libc++ hardening make this more likely to surface |
| 6 | **Build system.** waf to CMake, for 3 modules and the tests | Low | carlosrisma's `CMakeLists.txt` and the groupcast one for `traci` are reusable. It also removes the Python 2.7 dependency |
| 7 | **macOS.** Apple clang parsed C++23 here, but linking and `weak` behaviour are untested on new ns-3 | Low | Goes away with risk 2 |
| 8 | **Rebasing later.** Upstream's README said a port was underway. Its last push was March 2024 | Low | Ask the authors before starting, in case one exists |
| 9 | **EPC code.** `HIGHWAY` only uses it for a gateway address, and it holds the out-of-scope breaks | Low | Can be cut from the scenario |

## What does not need carrying over

Upstream altered 13 stock files. For Mode 2:

| Patch | Needed? |
|---|---|
| `propagation-loss-model.cc` | No. Debug files, including the 450 MB `.unimo` logs. `TwoRay` default frequency is unused, since MoReV2X has its own model |
| `multi-model-spectrum-channel.*` | No. Adds a `Gain` trace that never fires |
| `buildings-propagation-loss-model.*` | No. Debug prints |
| `callback.h` | No. Modern ns-3 callbacks take any number of arguments |
| `ns2-mobility-helper.h` | No. Unused |
| `udp-client.*`, `packet-sink` | Only for the override, see risk 2 |

## The options

| Option | Gain for Mode 2 | Cost | When it makes sense |
|---|---|---|---|
| **A. Stay on 3.22** | Everything is verified today | None now. Python 2.7 and a 2015 base stay | Your work is Mode 2 scheduling inside MoReV2X |
| **B. Port to 3.42** | **Same base as your groupcast tree**: compare MoReV2X and 5G-LENA Mode 2 on the same SUMO scenarios with one TraCI module | Smallest port, 48 errors | You want both simulators in one tree |
| **C. Port to 3.49** | Latest ns-3, no Python 2 | 22 more errors, C++23. **5G-LENA's sidelink stays behind**: its newest `v2x-v1.1` is tied to ns-3.42 | You need a feature of recent ns-3 |

**No option adds a Mode 2 feature.** New ns-3 gives no sidelink behaviour to MoReV2X, whose MAC and PHY are its own. Migrating buys maintainability and co-existence, not results.

**Recommendation:** stay on A while MoReV2X alone answers your questions. If you need to compare with 5G-LENA, do B. B to C later is a small step, so B does not close that door.

## If you go ahead

Each step goes through `tools/gate.sh`. Effort is a rough estimate of working days, and step 4 is the unknown.

| Step | Work | Effort |
|---|---|---|
| 1 | Baseline: `run_sweep.py` over loads and modes, 10+ seeds | 1, mostly compute |
| 2 | Replace the `UdpClient` override, on 3.22, proven by `fingerprint.sh` | 1 |
| 3 | Port: fixes above, CMake, tests, `debug-check.sh`, sanitizers | 2 to 3 |
| 4 | Validate against the baseline, and explain each difference | 2 to 5, open-ended if drift appears |

Steps 1 and 2 are useful on their own, and they are the moment to stop if the numbers argue against going on.

The next useful evidence is a real build: it needs a stock ns-3 checkout built on this machine, which goes beyond this read-only trial, so it needs your go-ahead.